In [1]:
import sys
sys.path.append("../src")

%load_ext autoreload
%autoreload 2

import pandas as pd
import numpy as np

raw = pd.read_parquet("../data/processed/full.parquet")
print(raw.shape)
raw.head(3)

(590540, 59)


,TransactionID,isFraud,TransactionDT,TransactionAmt,ProductCD,card1,card2,card3,card4,card5,...,M4,M5,M6,M7,M8,M9,day,week,split,batch
0,2987000,0,86400,68.5,W,13926,NaN,150.0,discover,142.0,...,M2,F,T,NaN,NaN,NaN,1.000000,0,train,-1
1,2987001,0,86401,29.0,W,2755,404.0,150.0,mastercard,102.0,...,M0,T,T,NaN,NaN,NaN,1.000012,0,train,-1
2,2987002,0,86469,59.0,W,4663,490.0,150.0,visa,166.0,...,M0,F,F,F,F,F,1.000799,0,train,-1


In [3]:
from features import build_features, to_categorical, META, FEATURES, ENG

feat = build_features(raw)
feat[ENG].head(10)

,hour,dayofweek,log_amt,amt_cents,email_match,n_missing,n_prior_txn,n_prior_24h,secs_since_last,amt_vs_uid_mean
0,0,1,4.241327,0.5,0,17,0,0,NaN,NaN
1,0,1,3.401197,0.0,0,20,0,0,NaN,NaN
2,0,1,4.094345,0.0,0,12,0,0,NaN,NaN
3,0,1,3.931826,0.0,0,17,0,0,NaN,NaN
4,0,1,3.931826,0.0,0,26,0,0,NaN,NaN
5,0,1,3.912023,0.0,0,15,0,0,NaN,NaN
6,0,1,5.075174,0.0,0,12,0,0,NaN,NaN
7,0,1,6.048553,0.5,0,20,0,0,NaN,NaN
8,0,1,2.772589,0.0,0,26,0,0,NaN,NaN
9,0,1,4.770685,0.0,0,12,0,0,NaN,NaN


In [4]:
feat[ENG].describe().T

,count,mean,std,min,25%,50%,75%,max
hour,590540.0,13.861923,7.607152e+00,0.000000,6.000000,16.00000,20.000000,2.300000e+01
dayofweek,590540.0,2.957940,2.033973e+00,0.000000,1.000000,3.00000,5.000000,6.000000e+00
log_amt,590540.0,4.382960,9.371828e-01,0.223943,3.791459,4.24519,4.836282,1.037156e+01
amt_cents,590540.0,0.379452,4.341180e-01,0.000000,0.000000,0.00000,0.949997,9.990005e-01
email_match,590540.0,0.173577,3.787454e-01,0.000000,0.000000,0.00000,0.000000,1.000000e+00
n_missing,590540.0,15.928909,4.675798e+00,9.000000,12.000000,15.00000,19.000000,3.100000e+01
n_prior_txn,590540.0,382.399658,9.775201e+02,0.000000,8.000000,46.00000,271.000000,9.899000e+03
n_prior_24h,590540.0,5.906387,2.400633e+01,0.000000,0.000000,1.00000,4.000000,8.800000e+02
secs_since_last,548868.0,371119.312500,1.081104e+06,0.000000,3051.000000,36125.50000,227485.250000,1.559967e+07
amt_vs_uid_mean,548868.0,1.147914,1.911316e+00,0.003986,0.414217,0.71672,1.214804,2.349848e+02


In [5]:
N = 200_000
small = build_features(raw.iloc[:N])

cols = ["n_missing", "hour", "dayofweek", "log_amt", "amt_cents", "email_match",
        "n_prior_txn", "n_prior_24h", "secs_since_last", "amt_vs_uid_mean"]

same = np.allclose(small[cols].to_numpy(dtype="float64"),
                   feat[cols].iloc[:N].to_numpy(dtype="float64"),
                   equal_nan=True)
print("Leak test passed:", same)
assert same

Leak test passed: True


In [6]:
feat = to_categorical(feat, feat["split"] == "train")
feat = feat[META + FEATURES]
print(feat.shape)
feat.dtypes.value_counts()

(590540, 69)


float32     41
category     8
int64        6
int8         4
int32        2
float64      1
str          1
category     1
category     1
category     1
category     1
category     1
category     1
Name: count, dtype: int64

In [7]:
num = feat[FEATURES].select_dtypes("number")
print("Infinite values:", np.isinf(num.to_numpy(dtype="float64")).sum())

Infinite values: 0


In [8]:
tr = feat[feat.split == "train"]

print(tr.groupby(tr.n_prior_txn == 0)["isFraud"].mean())
print(tr.groupby(pd.cut(tr.n_prior_24h, [-1, 0, 1, 3, 10, 1000]))["isFraud"].mean())
print(tr.groupby("email_match")["isFraud"].mean())

n_prior_txn
False    0.032603
True     0.020687
Name: isFraud, dtype: float64
n_prior_24h
(-1, 0]       0.021945
(0, 1]        0.031318
(1, 3]        0.036506
(3, 10]       0.043915
(10, 1000]    0.038350
Name: isFraud, dtype: float64
email_match
0    0.019878
1    0.070187
Name: isFraud, dtype: float64


In [9]:
print(tr.groupby(tr.n_prior_txn == 0)["isFraud"].agg(["size", "mean"]))
print(tr.groupby(pd.cut(tr.n_prior_24h, [-1, 0, 1, 3, 10, 1000]), observed=True)["isFraud"].agg(["size", "mean"]))
print(tr.groupby([tr.R_emaildomain.notna(), tr.email_match])["isFraud"].agg(["size", "mean"]))

               size      mean
n_prior_txn                  
False        193204  0.032603
True          27602  0.020687
              size      mean
n_prior_24h                 
(-1, 0]      91412  0.021945
(0, 1]       33335  0.031318
(1, 3]       30762  0.036506
(3, 10]      34658  0.043915
(10, 1000]   30639  0.038350
                             size      mean
R_emaildomain email_match                  
False         0            153312  0.019424
True          0             18183  0.023703
              1             49311  0.070187


Fraud rate in the training period is 3.1%. Customers with prior activity have a higher fraud rate (3.3%) than first-time pseudo-customers (2.1%), the opposite of my initial guess. Fraud rate rises with the number of payments in the previous 24 hours, from 2.2% with none to 4.4% with 4-10, suggesting burst activity is a signal. Rows where buyer and recipient emails match have a much higher fraud rate (7.0% vs 2.0%), email_match splits into three groups: missing recipient email (1.9% fraud), different emails (2.4%), and matching emails (7.0%). The signal comes from matching emails, not from the presence of a recipient email. These are univariate, train-only observations and not causal claims.

In [10]:
feat.to_parquet("../data/processed/features.parquet", index=False)

In [11]:
chk = pd.read_parquet("../data/processed/features.parquet")
print(chk.shape)
print(chk["ProductCD"].dtype)

(590540, 69)
category
